# Falsifying the Algorithm Prime Laws

AuthorityEffect: None. This notebook attempts to falsify or violate the invariant laws of the Algorithm Prime Triad: unadmitted measurement, broken declaration order, spurious continuation, and tamper of the Merkle tape. Each violation must be refused by typed exceptions or status codes.

In [ ]:
import sys
from fractions import Fraction
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "oe_prime_algorithms.md").is_file())
sys.path.insert(0, str(root / "code"))
import oe_prime_algorithms as opa
print("Algorithm Prime module loaded for adversarial falsification")

## 1. Falsification Attempt 1: Measurement Preceding Admission

Invariant: Measurement never precedes admission. Calling analyze_1cochain with default or admitted=False must raise PermissionError.

In [ ]:
carrier = opa.SimplicialCarrier(vertices=["a", "b"], edges=[("a", "b")])
h = {("a", "b"): Fraction(1)}

try:
    opa.ObstructionCalculus.analyze_1cochain(carrier, h, admitted=False)
    assert False, "Failed to refuse unadmitted measurement!"
except PermissionError as e:
    print("Admission firewall held:", e)
    assert "MEASUREMENT_REFUSED_UNADMITTED_PACKET" in str(e)

try:
    opa.ObstructionCalculus.analyze_1cochain(carrier, h)
    assert False, "Failed to refuse default unadmitted measurement!"
except PermissionError as e:
    print("Default firewall held:", e)
    assert "MEASUREMENT_REFUSED_UNADMITTED_PACKET" in str(e)

## 2. Falsification Attempt 2: Bypassing Local Declaration Order

Submitting candidate declarations with missing grounding, broken carriers, or missing witnesses must be rejected.

In [ ]:
obs = opa.ObserverPacket(observer_id="obs_bad", gauge_name="L1")
bad_decl = opa.LocalDeclarationOrder(
    step0_grounding_hash="",
    step1_carrier=carrier,
    step2_observer=obs,
    step3_primitive_difference=h,
    step4_zero_rel_rule="EXACT",
    step5_witness_certificate="WIT"
)
ok, reason = bad_decl.validate_declaration_order()
print("Grounding omission refused:", reason)
assert not ok and reason == "REFUSE_MISSING_GROUNDING"

## 3. Falsification Attempt 3: Spurious Continuation (Anti-Cheat Invariant)

Flipping any single continuation conjunct to False must halt continuation.

In [ ]:
engine = opa.PrimeTriadEngine(budget=100, delta_control=10)
# Missing obligation
c1 = engine.can_continue(0, False, True, True, True, True, True, True, True)
assert c1 is False
# Missing witness
c2 = engine.can_continue(0, True, False, True, True, True, True, True, True)
assert c2 is False
# GTLA refusal asserted
c3 = engine.can_continue(0, True, True, True, True, True, True, True, True, gtla_refusal=True)
assert c3 is False
print("All spurious continuation attempts halted successfully")

## 4. Falsification Attempt 4: Tampering with Merkle Tape

Modifying any receipt hash or outcome on the Merkle trace tape must be detected.

In [ ]:
terminalizer = opa.DDATerminalizer(engine, genesis_hash="GENESIS_ROOT")
terminalizer.process_step("E", {"action": "exact"})
terminalizer.process_step("R", {"action": "proj"})

# Tamper with the first entry hash
tampered_tape = [dict(entry) for entry in terminalizer.merkle_tape]
tampered_tape[0]["hash"] = "forged_hash_00000000000000000000000000000000"
ok_tamper, tamper_reason = opa.verify_merkle_tape(tampered_tape, "GENESIS_ROOT")
print("Tamper check detected:", tamper_reason)
assert not ok_tamper and tamper_reason == "ENTRY_HASH_MISMATCH"